# M8.1 · Mini-LLM: eğit, profil çıkar, hızlandır

Önceki fazların hepsini tek bir projede birleştiriyorsun: iki basamaklı toplamayı sıfırdan öğrenen küçük bir GPT. Hesap bütçesini (Chinchilla, 6ND FLOP) ve öğrenme oranı programını elle çıkarıyor, saf Python'da veri biçimini ve programı yazıyor, NumPy'da veri kümesini string üretmeden, tamamen vektörel kuruyor ve taban kayıpları hesaplıyor, PyTorch'ta warmup + cosine, gradyan kırpma ve maskeli kayıpla eğitip %95+ doğruluğa ulaşıyor ve en sonda profiler ile darboğazı bulup head döngüsünü tek bir batch'li işleme indiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m8-1-mini-llm.html](https://m-gonul.github.io/pytorch-lab/m8-1-mini-llm.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m8_1_mini_llm.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def lr():\n        f = need(ns, \"lr_schedule\")\n        close([f(0, 100, 1100, 1e-3, 1e-4), f(49, 100, 1100, 1e-3, 1e-4), f(99, 100, 1100, 1e-3, 1e-4)], [1e-5, 5e-4, 1e-3])\n        close([f(100, 100, 1100, 1e-3, 1e-4), f(600, 100, 1100, 1e-3, 1e-4), f(1100, 100, 1100, 1e-3, 1e-4), f(5000, 100, 1100, 1e-3, 1e-4)], [1e-3, 5.5e-4, 1e-4, 1e-4])\n        vals = [f(t, 100, 1100, 1e-3, 1e-4) for t in range(100, 1100)]\n        ensure(all(a >= b for a, b in zip(vals, vals[1:])), \"warmup'tan sonra monoton azalmalı\")\n    R.test(\"lr_schedule\", lr)\n    R.test(\"bütçe\", lambda: (ensure(need(ns, \"train_flops\")(1e7, 2e8) == 1.2e16, \"6ND\"), ensure(need(ns, \"chinchilla_tokens\")(1e7) == 2e8, \"20N\")))\n\n    def fmt():\n        ml = need(ns, \"make_line\")\n        ensure(ml(7, 45) == \"07+45=250\\n\" and ml(99, 99) == \"99+99=891\\n\" and ml(0, 0) == \"00+00=000\\n\", \"biçim\")\n        ensure(all(len(ml(a, b)) == 10 for a in range(0, 100, 7) for b in range(0, 100, 11)), \"her satır 10 karakter\")\n        pa = need(ns, \"parse_answer\")\n        ensure(all(pa(ml(a, b)[6:9]) == a + b for a in range(100) for b in range(0, 100, 13)), \"gidiş-dönüş\")\n    R.test(\"make_line / parse_answer\", fmt)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"make_dataset/encode döngüsüz\", lambda: (max_loops(src, 0, func=\"make_dataset\"), max_loops(src, 0, func=\"encode\")))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    V = \"0123456789+=\\n\"\n\n    def ds():\n        D = need(ns, \"make_dataset\")(500, np.random.default_rng(3))\n        ensure(D.shape == (500, 10), \"şekil (n, 10)\")\n        for row in D[:50]:\n            s = \"\".join(V[i] for i in row)\n            a, b = int(s[:2]), int(s[3:5])\n            ensure(s == f\"{a:02d}+{b:02d}={str(a + b).zfill(3)[::-1]}\\n\", f\"satır {s!r}\")\n    R.test(\"make_dataset = string biçimi\", ds)\n\n    def enc():\n        text = \"07+45=250\\n99+99=891\\n\"\n        ensure(np.array_equal(need(ns, \"encode\")(text), [[0, 7, 10, 4, 5, 11, 2, 5, 0, 12], [9, 9, 10, 9, 9, 11, 8, 9, 1, 12]]), \"kodlama\")\n    R.test(\"encode\", enc)\n\n    def ent():\n        D = ns[\"make_dataset\"](50000, np.random.default_rng(0))\n        H = need(ns, \"answer_entropy\")(D)\n        a = D[:, 0] * 10 + D[:, 1]; b = D[:, 3] * 10 + D[:, 4]; s = a + b\n        ref = 0.0\n        for col in (s % 10, (s // 10) % 10, s // 100):\n            p = np.bincount(col) / len(col); p = p[p > 0]\n            ref -= (p * np.log(p)).sum()\n        close(H, ref)\n        return f\"taban: {H:.3f} nat / satır (birler ve onlar ≈ log 10, yüzler ≈ 0.69). Model bunu ~0'a indirmeli.\"\n    R.test(\"answer_entropy (taban kayıp)\", ent)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    GPT, make_dataset = need(ns, \"GPT\", \"sınıf\"), need(ns, \"make_dataset\")\n\n    def lr():\n        f = need(ns, \"lr_at\")\n        close([f(49, 100, 1100, 1e-3, 1e-4), f(600, 100, 1100, 1e-3, 1e-4), f(1100, 100, 1100, 1e-3, 1e-4)], [5e-4, 5.5e-4, 1e-4])\n    R.test(\"lr_at\", lr)\n\n    def slv():\n        torch.manual_seed(0)\n        m = GPT().eval()\n        D = make_dataset(7, 1)\n        out = need(ns, \"solve\")(m, D[:, :6])\n        ensure(tuple(out.shape) == (7, 3), \"çıktı (B, 3)\")\n        idx = D[:, :6]\n        with torch.no_grad():\n            for _ in range(3):\n                idx = torch.cat([idx, m(idx)[:, -1].argmax(-1, keepdim=True)], 1)\n        ensure(torch.equal(out, idx[:, -3:]), \"greedy çözme\")\n    R.test(\"solve (greedy, batch'li)\", slv)\n\n    def tr():\n        torch.manual_seed(0)\n        m = GPT()\n        loss = need(ns, \"train\")(m, make_dataset(20000, 0), steps=1500)\n        acc = need(ns, \"accuracy\")(m, make_dataset(2000, 99))\n        ensure(acc >= 0.95, f\"test doğruluğu {acc:.3f}\")\n        n = sum(p.numel() for p in m.parameters())\n        return f\"{n:,} parametre, son kayıp {loss:.4f}, görülmemiş 2000 toplamada doğruluk {acc:.3f}\"\n    R.test(\"eğitim: iki basamaklı toplama\", tr)\n\n    def unseen():\n        D = make_dataset(20000, 0)\n        a = D[:, 0] * 10 + D[:, 1]; b = D[:, 3] * 10 + D[:, 4]\n        seen = set((a * 100 + b).tolist())\n        return f\"eğitim kümesi 10,000 olası çiftin {len(seen):,} tanesini içeriyor; test çiftleri büyük ölçüde örtüşüyor, genelleme için hold-out çiftlerle de dene\"\n    R.test(\"not: veri örtüşmesi\", unseen)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"FusedAttention.forward'da döngü yok\", lambda: max_loops(src, 0, func=\"forward\") if False else _no_loop_in_fused(src))\n    Loop = need(ns, \"LoopAttention\", \"sınıf\")\n    torch.manual_seed(0)\n    loop = Loop()\n    x = torch.randn(16, 64, 128)\n\n    def tops():\n        names = need(ns, \"top_ops\")(lambda: loop(x), 5)\n        ensure(isinstance(names, list) and len(names) == 5 and all(isinstance(n, str) for n in names), \"5 işlem adı\")\n        ensure(any(n.startswith(\"aten::\") for n in names), \"aten işlem adları bekleniyordu\")\n        return \"en pahalı işlemler: \" + \", \".join(names)\n    R.test(\"top_ops (profiler)\", tops)\n\n    def eq():\n        f = need(ns, \"FusedAttention\", \"sınıf\").from_loop(loop)\n        with torch.no_grad():\n            close(f(x), loop(x), atol=1e-5)\n        ensure(sum(isinstance(m, torch.nn.Linear) for m in f.modules()) == 2, \"tek qkv + proj: 2 Linear\")\n    R.test(\"FusedAttention = LoopAttention\", eq)\n    if \"FusedAttention\" not in ns:\n        return\n    f = ns[\"FusedAttention\"].from_loop(loop)\n\n    def step(m):\n        m.zero_grad()\n        m(x).pow(2).mean().backward()\n    t_loop = bench(lambda: step(loop), repeat=5)\n    t_fused = bench(lambda: step(f), repeat=5)\n    R.table(\"İleri + geri, B = 16, T = 64, d = 128, 8 head\", [\"\", \"süre\"], [[\"head döngüsü\", fmt_time(t_loop)], [\"birleşik\", fmt_time(t_fused)]],\n            note=f\"{t_loop / t_fused:.1f}× — 24 küçük matmul yerine 1 büyük matmul + fused attention.\")\n\n\ndef _no_loop_in_fused(src):\n    import ast\n    for node in ast.walk(ast.parse(src)):\n        if isinstance(node, ast.ClassDef) and node.name == \"FusedAttention\":\n            for fn in node.body:\n                if isinstance(fn, ast.FunctionDef) and fn.name == \"forward\":\n                    for sub in ast.walk(fn):\n                        if isinstance(sub, (ast.For, ast.While, ast.ListComp, ast.GeneratorExp)):\n                            raise AssertionError(\"FusedAttention.forward içinde döngü var\")\n            return\n    raise AssertionError(\"FusedAttention sınıfı yok\")\n"
}

In [ ]:
import importlib, labcheck, checks_m8_1_mini_llm as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bu modül bir mini proje: minGPT'nin "adder" örneğinden esinlenen, iki basamaklı toplamayı öğrenen bir GPT.

**Veri:** her örnek bir satır, `ab+cd=xyz⏎`; cevap 3 basamağa sıfırla doldurulup **ters çevrilir** (52 → `250`). Sözlük: `0123456789+=⏎` (13 token). Model girdi olarak ilk 9 token'ı görür, her konumda sonrakini tahmin eder; **kayıp yalnız cevap basamaklarını** tahmin eden 3 konumda.

**Eğitim döngüsü** (önceki modüllerden):

- AdamW (M3.4), öğrenme oranı: doğrusal warmup, sonra cosine düşüş:

$$
\eta_t=\begin{cases}\eta_{\max}\frac{t+1}{W} & t<W\\[2pt] \eta_{\min}+\tfrac12(\eta_{\max}-\eta_{\min})\big(1+\cos\pi\frac{t-W}{T-W}\big) & t\ge W\end{cases}
$$

- gradyan normu kırpma (`clip_grad_norm_`), değerlendirmede greedy çözme (M4.7).

**Bütçe:** yoğun bir Transformer'ın eğitimi $\approx6ND$ FLOP ($N$ parametre, $D$ token: ileri $2ND$, geri $4ND$). Chinchilla'ya göre hesap-optimal $D\approx20N$.

**Profil:** `torch.profiler.profile` hangi işlemin ne kadar sürdüğünü gösterir; çok sayıda küçük işlem (örneğin head başına ayrı matmul) Python ve kernel başlatma yükü demektir.

## 1 · Elle

MFU (model FLOP utilization): gerçekte kullanılan FLOP/s'nin donanım tepesine oranı; iyi ayarlanmış büyük eğitimlerde %40–60.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Bütçe: $N=10^7$ parametre**

- `chin`: Chinchilla-optimal token sayısı ($\approx20N$)
- `flops`: Eğitim FLOP'u $\approx6ND$
- `secs`: 100 TFLOP/s tepe, %40 MFU ile süre (s)

**Warmup 100 adım, toplam 1100, $\eta_{\max}=10^{-3}$, $\eta_{\min}=10^{-4}$, cosine**

- `lr50`: Adım 49'da $\eta$ (doğrusal warmup: $\eta_{\max}(t+1)/W$)
- `lr600`: Adım 600'de $\eta=\eta_{\min}+\tfrac12(\eta_{\max}-\eta_{\min})(1+\cos\pi p)$

**Biçim: `07+45=250⏎` (cevap 052 ters çevrilmiş), 10 karakter**

- `ans`: `38+67=` için ters çevrilmiş 3 basamaklı cevap (sayı olarak)
- `tok`: Batch 128: adım başına girdi token'ı (satır başına 9)

In [ ]:
cevaplar = dict(
    chin=None,
    flops=None,
    secs=None,
    lr50=None,
    lr600=None,
    ans=None,
    tok=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"chin":[200000000,0],"flops":[12000000000000000,10000000000],"secs":[300,0],"lr50":[0.0005,1e-7],"lr600":[0.00055,1e-7],"ans":[501,0],"tok":[1152,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `lr_schedule(step, warmup, total, max_lr, min_lr)` — yukarıdaki formül; `step ≥ total` için `min_lr`.
- `train_flops(N, D)` ve `chinchilla_tokens(N)`.
- `make_line(a, b)` → `"07+45=250\n"`.
- `parse_answer(s)` — modelin ürettiği 3 karakterlik ters cevabı sayıya çevir (`"250"` → 52).

In [ ]:
%%writefile stage_b.py
import math


def lr_schedule(step, warmup, total, max_lr, min_lr):
    ...


def train_flops(N, D):
    ...


def chinchilla_tokens(N):
    ...


def make_line(a, b):
    ...


def parse_answer(s):
    ...


print(lr_schedule(600, 100, 1100, 1e-3, 1e-4))     # 5.5e-4
print(make_line(7, 45), parse_answer("250"))        # 07+45=250 / 52

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**. Token kimlikleri: `0..9` rakamlar, `10` = `+`, `11` = `=`, `12` = satır sonu.

- `make_dataset(n, rng)` → `(n, 10)` int dizi; `a, b = rng.integers(0, 100, (2, n))`; rakamları `//` ve `%` ile çıkar, string üretme.
- `encode(text)` → `(satır, 10)`; arama tablosu: `lut[np.frombuffer(text.encode(), np.uint8)]`.
- `answer_entropy(D)` → cevap konumlarının (6, 7, 8) marjinal dağılımlarının entropileri toplamı (nat). Model bundan iyi olmalı; "sabit tahmin" tabanı budur. `np.bincount` ile.

In [ ]:
%%writefile stage_c.py
import numpy as np

VOCAB = "0123456789+=\n"


def make_dataset(n, rng):
    ...


def encode(text):
    ...


def answer_entropy(D):
    ...


D = make_dataset(5, np.random.default_rng(0))
print(D)
print(encode("07+45=250\n"))
print(answer_entropy(make_dataset(20000, np.random.default_rng(0))))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`GPT` hazır (M4.7'deki yapı, küçük başlatmalı). Sen yaz:

- `lr_at(step, warmup, total, max_lr, min_lr)` — b'deki program.
- `train(model, D, steps=1500, batch=128, max_lr=3e-3, seed=0)` — `torch.Generator(seed)` ile rastgele satırlar; `x = D[ix, :-1]`, `y = D[ix, 1:]`; kayıp yalnız `[:, 5:8]` konumlarında; AdamW (`weight_decay=0.01`), warmup 100 adım, `min_lr = max_lr / 10`, `clip_grad_norm_(…, 1.0)`. Son kaybı döndür.
- `solve(model, prompts)` — `prompts: (B, 6)` (`ab+cd=`), 3 adım greedy, `(B, 3)` cevap token'ları. Batch üzerinde vektörel.
- `accuracy(model, D)` — 3 basamağın tamamı doğru olan satırların oranı.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F
from torch import nn

VOCAB = "0123456789+=\n"


class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(d, -1)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, d))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, V=13, T=10, d=64, L=2, h=4):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, d), nn.Embedding(T, d)
        self.blocks = nn.ModuleList(Block(d, h) for _ in range(L))
        self.ln = nn.LayerNorm(d)
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, idx):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1]))
        for b in self.blocks:
            x = b(x)
        return self.ln(x) @ self.wte.weight.T


def make_dataset(n, seed):
    g = torch.Generator().manual_seed(seed)
    a, b = torch.randint(0, 100, (2, n), generator=g)
    s = a + b
    return torch.stack([a // 10, a % 10, torch.full((n,), 10), b // 10, b % 10, torch.full((n,), 11),
                        s % 10, (s // 10) % 10, s // 100, torch.full((n,), 12)], 1)


# ---- senin kodun ----

def lr_at(step, warmup, total, max_lr, min_lr):
    ...


def train(model, D, steps=1500, batch=128, max_lr=3e-3, seed=0):
    ...


@torch.no_grad()
def solve(model, prompts):
    ...


def accuracy(model, D):
    ...


torch.manual_seed(0)
model = GPT()
print(train(model, make_dataset(20000, 0), steps=300))
print(accuracy(model, make_dataset(1000, 99)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Profil çıkar ve hızlandır.

`LoopAttention` hazır: head başına ayrı `q`, `k`, `v` `nn.Linear`'ları ve head'ler üzerinde bir Python döngüsü. Sen yaz:

- `top_ops(fn, k=5)` — `torch.profiler.profile(activities=[ProfilerActivity.CPU])` altında `fn()`'i çalıştır; `key_averages()`'ı `self_cpu_time_total`'a göre sırala, ilk `k` işlemin adını döndür.
- `FusedAttention.from_loop(loop)` — tek bir `qkv = nn.Linear(d, 3d)`; ağırlıkları head'lerin ağırlıklarını **yığarak** kur (çıktı sırası: tüm q head'leri, tüm k head'leri, tüm v head'leri), aynı `proj`. `forward(x)`: tek matmul + reshape + `F.scaled_dot_product_attention(is_causal=True)`. **Head döngüsü yok.**

Kontrol çıktıların aynı olduğunu ve hız kazancını ölçer.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F
from torch import nn
from torch.profiler import profile, ProfilerActivity


class LoopAttention(nn.Module):
    """Yavaş başvuru: head başına ayrı Linear'lar, head'ler üzerinde döngü."""
    def __init__(self, d=128, h=8):
        super().__init__()
        self.h, self.dk = h, d // h
        self.q = nn.ModuleList(nn.Linear(d, self.dk) for _ in range(h))
        self.k = nn.ModuleList(nn.Linear(d, self.dk) for _ in range(h))
        self.v = nn.ModuleList(nn.Linear(d, self.dk) for _ in range(h))
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        T = x.shape[1]
        mask = torch.ones(T, T, dtype=torch.bool).triu(1)
        outs = []
        for i in range(self.h):
            q, k, v = self.q[i](x), self.k[i](x), self.v[i](x)
            att = (q @ k.transpose(-2, -1)) / self.dk ** 0.5
            att = att.masked_fill(mask, float("-inf")).softmax(-1)
            outs.append(att @ v)
        return self.proj(torch.cat(outs, -1))


# ---- senin kodun ----

def top_ops(fn, k=5):
    ...


class FusedAttention(nn.Module):
    @classmethod
    def from_loop(cls, loop):
        ...

    def forward(self, x):
        ...


torch.manual_seed(0)
loop = LoopAttention()
x = torch.randn(16, 64, 128)
print(top_ops(lambda: loop(x)))
fused = FusedAttention.from_loop(loop)
print((fused(x) - loop(x)).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Touvron ve ark. (2023): LLaMA

[Touvron ve ark. — LLaMA: Open and Efficient Foundation Language Models (2023)](https://arxiv.org/abs/2302.13971)

Mini-LLM'ine LLaMA'nın mimari değişikliklerini (Bölüm 2.2) tek tek ekleyip ablasyon yap: RMSNorm (Pre-norm), SwiGLU FFN (gizli boyut 2/3·4d), RoPE (M4.14 x1), bias'sız doğrusal katmanlar. Aynı hesap bütçesiyle doğrulama kaybını ve token/s'yi karşılaştır; KV-cache'li üretimde RoPE'nin konum kaydırmayla nasıl çalıştığını göster.


**Adımlar**

1. SwiGLU'da parametre sayısını eşitlemek için gizli boyutu ayarla.

**Kabul ölçütleri**

- Her değişikliğin kayba etkisi tablosu.
- LLaMA tarzı model aynı bütçede GPT-2 tarzından iyi ya da eşit.

## Makale ödevi: Hoffmann ve ark. (2022): hesap-optimal LLM eğitimi (Chinchilla)

[Hoffmann ve ark. — Training Compute-Optimal Large Language Models (NeurIPS 2022)](https://arxiv.org/abs/2203.15556)

Makalenin 1. yaklaşımını (sabit model boyutları, farklı token sayıları) küçük ölçekte uygula: 4–6 model boyutu × farklı eğitim uzunlukları; her FLOP bütçesi için minimum kaybı veren (N, D)'yi bul ve N_opt ∝ C^a, D_opt ∝ C^b üslerini uydur (makale: a ≈ b ≈ 0.5). Parametrik kayıp formunu L(N, D) = E + A/N^α + B/D^β (Yaklaşım 3) ölçümlerine uydur.


**Adımlar**

1. FLOP ≈ 6ND; öğrenme oranı kosinüs takvimini eğitim uzunluğuna eşitle (makalenin kritik bulgusu).

**Kabul ölçütleri**

- a ve b üsleri raporlanmış, yaklaşık eşit.
- Parametrik form uydurması ve kalıntılar.

## Kaynaklar

- [Karpathy — nanoGPT ve minGPT (adder örneği)](https://github.com/karpathy/minGPT) — MIT
- [Stanford CS336 — Language Modeling from Scratch (Assignment 1)](https://github.com/stanford-cs336/assignment1-basics) — MIT
- [Hoffmann ve ark. — Training Compute-Optimal Large Language Models (Chinchilla)](https://arxiv.org/abs/2203.15556) — arXiv
- [Kaplan ve ark. — Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) — arXiv
- [Lee ve ark. — Teaching Arithmetic to Small Transformers (ters çevrilmiş çıktı)](https://arxiv.org/abs/2307.03381) — arXiv
- [PyTorch Tutorials — Profiling your PyTorch Module](https://docs.pytorch.org/tutorials/beginner/profiler.html) — BSD-3-Clause